# Sensitivity analysis: alternative CO2e thresholds



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

# =====================================================
# Load data for all cities and modes (same pipeline as 12_1)
# =====================================================

cities = ["Helsinki", "Turku", "Tampere", "Oulu"]
modes = ["Cycling", "PT", "Car"]

data_list = []
for city in cities:
    for mode in modes:
        if mode == "Cycling":
            df = pd.read_parquet(f"./output/tour_bike_{city.lower()}_decent_per_user.parquet")
        elif mode == "PT":
            df = pd.read_parquet(f"./output/tour_pt_{city.lower()}_decent_per_user.parquet")
        elif mode == "Car":
            df = pd.read_parquet(f"./output/tour_car_{city.lower()}_decent_per_user.parquet")

        co2_values = df["decent_mobility_co2"] / 1000  # kg

        for co2 in co2_values:
            data_list.append({"City": city, "Mode": mode, "CO2": co2})

plot_df = pd.DataFrame(data_list)
plot_df["CO2"] = pd.to_numeric(plot_df["CO2"], errors="coerce")
plot_df = plot_df.dropna(subset=["CO2"])
plot_df["CO2"] = np.clip(plot_df["CO2"], 0, None)


In [ ]:
# =====================================================
# Sensitivity thresholds
# 3.8 kg = stringent threshold requested by coauthor
# 5.6 / 8.4 kg = -20% / +20% computed over the original 7 kg budget
# =====================================================

THRESHOLDS = {
    "Stringent (3.8 kg)": 3.8,
    "-20% (5.6 kg)": 7.0 * 0.8,
    "Baseline (7 kg)": 7.0,
    "+20% (8.4 kg)": 7.0 * 1.2,
}

mode_colors = {
    "Cycling": "#9FACCA",   # A2 - dusty blue-grey
    "PT": "#574E9C",        # B3 - deep indigo
    "Car": "#9A2588",       # C2 - muted plum-magenta
}


## Compliance table across thresholds

Extends the Panel B `% <= threshold` table (originally computed only at 7 kg)
to all four thresholds, per city and mode.

In [ ]:
rows = []
for city in cities:
    for mode in modes:
        subset = plot_df[(plot_df["City"] == city) & (plot_df["Mode"] == mode)]
        for label, thr in THRESHOLDS.items():
            pct = np.nan if len(subset) == 0 else (subset["CO2"] <= thr).mean() * 100
            rows.append({
                "City": city,
                "Mode": mode,
                "Threshold": label,
                "Threshold_kg": thr,
                "% compliant": round(pct, 1),
            })

sensitivity_table = pd.DataFrame(rows)
sensitivity_pivot = sensitivity_table.pivot_table(
    index=["City", "Mode"], columns="Threshold", values="% compliant"
)[list(THRESHOLDS.keys())]

sensitivity_pivot


## Panel A extended: boxplot with four threshold lines

Same horizontal boxplot as `12_1` Panel A (identical styling, Nature rcParams,
mode colours), with all four thresholds drawn instead of just the 7 kg line.

In [ ]:
# ---------------------------------------------------------------
# Order and data (same construction as 12_1 Panel A)
# ---------------------------------------------------------------
tick_label_map = {"Cycling": "Cycling", "PT": "PT", "Car": "Car"}

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 8,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 2.5,
    "ytick.major.size": 2.5,
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
})

box_data, positions, tick_labels, colors, city_centers = [], [], [], [], []
gap = 1.6
y = 0.1
for city in cities:
    city_positions = []
    for mode in modes:
        values = (
            plot_df.loc[(plot_df["City"] == city) & (plot_df["Mode"] == mode), "CO2"]
            .astype(float).values
        )
        values = np.clip(values, 0, None)
        box_data.append(values)
        positions.append(y)
        tick_labels.append(tick_label_map[mode])
        colors.append(mode_colors[mode])
        city_positions.append(y)
        y += 1
    city_centers.append(np.mean(city_positions))
    y += gap

fig, ax = plt.subplots(figsize=(7.2, 6.0), dpi=300)

bp = ax.boxplot(
    box_data, positions=positions, vert=False, widths=0.8,
    patch_artist=True, showfliers=False,
)

for box, c in zip(bp["boxes"], colors):
    box.set_facecolor(c)
    box.set_alpha(0.9)
    box.set_edgecolor("#333333")
    box.set_linewidth(0.9)

for whisker in bp["whiskers"]:
    whisker.set_color("#444444")
    whisker.set_linewidth(0.7)

for cap in bp["caps"]:
    cap.set_color("#444444")
    cap.set_linewidth(0.7)

for median in bp["medians"]:
    median.set_color("white")
    median.set_linewidth(1.8)

# ---------------------------------------------------------------
# Threshold lines (distinguished by linestyle, not colour)
# ---------------------------------------------------------------
threshold_styles = {
    "Stringent (3.8 kg)": (0, (1, 1)),
    "-20% (5.6 kg)": (0, (2, 2)),
    "Baseline (7 kg)": (0, (4, 3)),
    "+20% (8.4 kg)": (0, (6, 2, 1, 2)),
}

for label, thr in THRESHOLDS.items():
    ax.axvline(thr, color="#333333", linestyle=threshold_styles[label], linewidth=1.1, zorder=1)

ax.set_yticks(positions)
ax.set_yticklabels(tick_labels, fontsize=9)

for city, c in zip(cities, city_centers):
    ax.text(
        -0.08, c, city, transform=ax.get_yaxis_transform(),
        ha="right", va="center", fontsize=9, fontweight="bold", color="#111111",
    )

ax.set_xlabel("CO$_2$ (kg/week per person)", fontsize=9, labelpad=4)
ax.set_ylabel("")

upper = plot_df["CO2"].quantile(0.995)
ax.set_xlim(0, upper * 1.1)

ax.xaxis.grid(True, color="#E5E5E5", linewidth=0.6, zorder=0)
ax.set_axisbelow(True)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_color("#444444")

ax.tick_params(axis="y", length=0)
ax.tick_params(axis="x", color="#444444")

ax.invert_yaxis()

handles = [
    mpatches.Patch(facecolor=mode_colors[m], edgecolor="#333333", alpha=0.9, label=tick_label_map[m])
    for m in modes
]
for label in THRESHOLDS:
    handles.append(Line2D([0], [0], color="#333333", linestyle=threshold_styles[label], lw=1.1, label=label))

legend = ax.legend(
    handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.1),
    ncol=4, frameon=False, fontsize=8, handlelength=1.4, columnspacing=1.2,
)

plt.tight_layout()

out_png = "./output/co2_boxplot_panelA_sensitivity.png"
out_svg = "./output/co2_boxplot_panelA_sensitivity.svg"
plt.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
plt.savefig(out_svg, bbox_inches="tight", facecolor="white")
print("Saved:", out_png, out_svg)


## Compliance vs threshold, by city and mode

Compact view of how compliance shifts as the threshold moves from
stringent (3.8 kg) to +20% (8.4 kg), one panel per city.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 13), sharey=True)
axes = axes.flatten()

x_labels = list(THRESHOLDS.keys())
x_vals = list(THRESHOLDS.values())

for ax, city in zip(axes, cities):
    for mode in modes:
        sub = sensitivity_table[
            (sensitivity_table["City"] == city) & (sensitivity_table["Mode"] == mode)
        ].set_index("Threshold").loc[x_labels]

        ax.plot(
            x_vals, sub["% compliant"],
            marker="o", linewidth=3.5, markersize=10,
            color=mode_colors[mode], label=mode,
            clip_on=False,
        )

    ax.set_title(city, fontsize=20, fontweight="bold", pad=14)
    ax.set_xticks(x_vals)
    ax.set_xticklabels([f"{v:.1f}" for v in x_vals], fontsize=15)
    ax.tick_params(axis="y", labelsize=15)
    ax.set_xlabel("Threshold (kg CO$_2$e/week)", fontsize=16)
    ax.axvline(7.0, color="#999999", linestyle=":", linewidth=1.5, zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.2)
    ax.spines["bottom"].set_linewidth(1.2)
    ax.grid(axis="y", color="#E5E5E5", linewidth=0.8)
    ax.set_axisbelow(True)

# extra headroom so points sitting at/near 100% aren't clipped by the axis
axes[0].set_ylabel("% compliant", fontsize=17)
axes[2].set_ylabel("% compliant", fontsize=17)
axes[0].set_ylim(-3, 108)

handles = [mpatches.Patch(facecolor=mode_colors[m], label=m) for m in modes]
fig.legend(
    handles=handles, loc="lower center", bbox_to_anchor=(0.5, -0.04),
    ncol=3, frameon=False, fontsize=17,
)

fig.suptitle(
    "Decent-mobility compliance across threshold sensitivity range",
    y=1.02, fontsize=20, fontweight="bold",
)

plt.tight_layout(rect=[0, 0.03, 1, 1])
out_png = "./output/compliance_sensitivity_by_threshold.png"
plt.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
print("Saved:", out_png)
plt.show()


## Compliance maps: inside vs outside the weekly budget

Hex-level version of the compliance table. Each home hexagon is classified with
two colours only - within budget / over budget - using the mean weekly CO2e of
the users living in it, and the classification is repeated for all four
thresholds so the maps show how the compliant area shrinks or grows.

Colours are taken from the project bivariate palette (B1 -> C3), consistent with
the hex maps in `12_2`. Home hexagons and the per-city spatial filters come from
the same sources as `12_2`.


In [ ]:
# =====================================================
# Data for the compliance maps
# Reuses the home-hexagon pipeline from 06_figures/fig3_income_inequality.ipynb:
#   - per-user weekly CO2e, keeping user_id (the first cell drops it)
#   - home hexagon per user from the user_pois_pt_1* files
#   - per-city spatial filter (Helsinki passes through, as in fig3_income_inequality.ipynb)
# =====================================================
import geopandas as gpd
import contextily as cx; import basemaps
import h3
from shapely.geometry import Polygon

# ---------------------------------------------------------------
# CARTO basemap tiles now require an API key. Unauthenticated
# requests still return HTTP 200, but the PNG carries a diagonal
# "API KEY REQUIRED" watermark, so nothing raises and the figures
# look fine until zoomed in. Free key (5M tiles/month):
#   https://carto.com/basemaps/apikey
# Set it in the environment before launching Jupyter, e.g.
#   export CARTO_API_KEY=...
# Keep the key out of the repo - do not paste it into this notebook.
# ---------------------------------------------------------------
import os
import warnings
from xyzservices import TileProvider

BASEMAP = TileProvider(basemaps.POSITRON)  # copy, leave the global alone
_carto_key = os.environ.get("CARTO_API_KEY", "")
if _carto_key:
    BASEMAP["url"] = BASEMAP["url"] + "?key={key}"
    BASEMAP["key"] = _carto_key
else:
    warnings.warn(
        "CARTO_API_KEY is not set: basemap tiles will carry the "
        "'API KEY REQUIRED' watermark. Get a free key at "
        "https://carto.com/basemaps/apikey",
        stacklevel=2,
    )

mode_files = {"Cycling": "bike", "PT": "pt", "Car": "car"}

per_user = {}
for city in cities:
    for mode in modes:
        df = pd.read_parquet(
            f"./output/tour_{mode_files[mode]}_{city.lower()}_decent_per_user.parquet"
        )[["user_id", "decent_mobility_co2"]].copy()
        df["co2_kg"] = pd.to_numeric(df["decent_mobility_co2"], errors="coerce") / 1000
        df["co2_kg"] = np.clip(df["co2_kg"], 0, None)
        per_user[(city, mode)] = df.dropna(subset=["co2_kg"])

# ---------------------------------------------------------------
# Home hexagon per user
# ---------------------------------------------------------------
city_file_map = {
    "Helsinki": "./data/user_pois_pt_1.parquet",
    "Turku":    "./data/user_pois_pt_1_turku.parquet",
    "Tampere":  "./data/user_pois_pt_1_tampere.parquet",
    "Oulu":     "./data/user_pois_pt_1_oulu.parquet",
}

home_frames = []
for city, path in city_file_map.items():
    df = pd.read_parquet(path)
    df = df[df["is_home"] == 1][["user_id", "home_gid9"]].copy()
    df["city"] = city
    home_frames.append(df)

# dedupe within a city, so a user appearing in two city files keeps the home
# hexagon relevant to each city's own CO2 file
user_home_hex = (
    pd.concat(home_frames, ignore_index=True)
    .dropna(subset=["home_gid9"])
    .drop_duplicates(subset=["user_id", "city"])
    .rename(columns={"home_gid9": "home_h3"})
)

# ---------------------------------------------------------------
# Spatial filters (same files as in fig3_income_inequality.ipynb; Helsinki unfiltered there too)
# ---------------------------------------------------------------
city_filter_map = {
    "Turku":   "./data/filter_hex_turku.geojson",
    "Tampere": "./data/filter_hex_tampere.geojson",
    "Oulu":    "./data/filter_hex_oulu.geojson",
}

city_boundaries = {
    city: gpd.read_file(path).to_crs("EPSG:3067").union_all()
    for city, path in city_filter_map.items()
}

def hex_to_polygon(hex_id):
    # h3 v3 API, returns (lng, lat) pairs
    return Polygon(h3.h3_to_geo_boundary(hex_id, geo_json=True))

# ---------------------------------------------------------------
# Hexagon-level aggregation, one GeoDataFrame per city and mode
# ---------------------------------------------------------------
MIN_USERS = 1   # raise (e.g. 3) to drop hexagons backed by very few users

hex_gdfs = {}
for city in cities:
    homes = user_home_hex[user_home_hex["city"] == city]
    for mode in modes:
        merged = homes.merge(per_user[(city, mode)], on="user_id", how="inner")

        hex_agg = (
            merged.groupby("home_h3")
            .agg(
                mean_co2=("co2_kg", "mean"),
                median_co2=("co2_kg", "median"),
                n_users=("user_id", "count"),
            )
            .reset_index()
        )
        hex_agg = hex_agg[hex_agg["n_users"] >= MIN_USERS]
        hex_agg["geometry"] = hex_agg["home_h3"].apply(hex_to_polygon)

        gdf = gpd.GeoDataFrame(
            hex_agg, geometry="geometry", crs="EPSG:4326"
        ).to_crs("EPSG:3067")

        if city in city_boundaries:
            gdf = gdf[gdf.geometry.centroid.within(city_boundaries[city])].copy()

        hex_gdfs[(city, mode)] = gdf
        print(f"{city:9s} {mode:8s} {len(gdf):5d} hexagons  "
              f"{merged['user_id'].nunique():6d} users")


In [ ]:
# =====================================================
# Binary compliance maps: 3 modes (rows) x 4 thresholds (columns), per city
# Two colours only, from the 12_2 bivariate palette
# =====================================================
import matplotlib.patches as mpatches

compliance_colors = {
    "inside":  "#9ecae1",   # A3 - blue, within the weekly budget
    "outside": "#7232A9",   # C3 - deep purple, over the weekly budget
}


HEX_METRIC = "mean_co2"   # or "median_co2" for a less outlier-sensitive hexagon value

def compliance_share(gdf, thr):
    """Share of hexagons within budget, %."""
    if len(gdf) == 0:
        return np.nan
    return (gdf[HEX_METRIC] <= thr).mean() * 100

def plot_city_compliance_maps(city, save=True):
    city_gdfs = {mode: hex_gdfs[(city, mode)] for mode in modes}
    drawn = [g for g in city_gdfs.values() if len(g) > 0]
    if not drawn:
        print(f"{city}: no hexagons to draw")
        return

    # shared extent across all panels of this city
    bounds = np.array([g.total_bounds for g in drawn])
    xmin, ymin = bounds[:, 0].min(), bounds[:, 1].min()
    xmax, ymax = bounds[:, 2].max(), bounds[:, 3].max()
    padx, pady = 0.03 * (xmax - xmin), 0.03 * (ymax - ymin)

    thr_labels = list(THRESHOLDS.keys())

    fig, axes = plt.subplots(
        len(modes), len(thr_labels),
        figsize=(4.2 * len(thr_labels), 4.6 * len(modes)),
    )
    axes = np.atleast_2d(axes)

    for i, mode in enumerate(modes):
        gdf = city_gdfs[mode]
        for j, label in enumerate(thr_labels):
            thr = THRESHOLDS[label]
            ax = axes[i, j]

            if len(gdf) > 0:
                gdf.plot(
                    ax=ax,
                    color=np.where(
                        gdf[HEX_METRIC] <= thr,
                        compliance_colors["inside"],
                        compliance_colors["outside"],
                    ),
                    edgecolor="#333333",
                    linewidth=0.2,
                    alpha=0.92,
                )

            ax.set_xlim(xmin - padx, xmax + padx)
            ax.set_ylim(ymin - pady, ymax + pady)
            cx.add_basemap(
                ax, crs="EPSG:3067",
                source=BASEMAP,
            )
            ax.set_axis_off()

            if i == 0:
                ax.set_title(label, fontsize=17, pad=10)

            if j == 0:
                ax.text(
                    -0.04, 0.5, mode,
                    transform=ax.transAxes, rotation=90,
                    ha="center", va="center", fontsize=17, fontweight="bold",
                )

    handles = [
        mpatches.Patch(facecolor=compliance_colors["inside"], edgecolor="#333333",
                       linewidth=0.4, label="Within weekly budget"),
        mpatches.Patch(facecolor=compliance_colors["outside"], edgecolor="#333333",
                       linewidth=0.4, label="Over weekly budget"),
    ]
    fig.legend(
        handles=handles, loc="lower center", ncol=2,
        bbox_to_anchor=(0.5, 0.005), frameon=False,
        fontsize=15,
    )

    fig.suptitle(
        f"{city}: home hexagons inside vs outside the weekly CO$_2$e budget "
        f"({HEX_METRIC.replace('_', ' ')} per hexagon)",
        fontsize=20, y=0.995,
    )
    fig.tight_layout(rect=[0.01, 0.04, 1, 0.98])

    if save:
        out = f"./output/{city.lower()}_compliance_maps_thresholds.png"
        fig.savefig(out, dpi=300, bbox_inches="tight", facecolor="white")
        print("saved", out)

    plt.show()

for city in cities:
    plot_city_compliance_maps(city)


In [ ]:
# =====================================================
# Hexagon-level companion to the user-level compliance table
# (% of home hexagons within budget, per city / mode / threshold)
# =====================================================
hex_rows = []
for city in cities:
    for mode in modes:
        gdf = hex_gdfs[(city, mode)]
        for label, thr in THRESHOLDS.items():
            hex_rows.append({
                "City": city,
                "Mode": mode,
                "Threshold": label,
                "% hexagons within budget": round(compliance_share(gdf, thr), 1),
                "n_hexagons": len(gdf),
            })

hex_sensitivity_table = pd.DataFrame(hex_rows)
hex_sensitivity_pivot = hex_sensitivity_table.pivot_table(
    index=["City", "Mode"], columns="Threshold", values="% hexagons within budget"
)[list(THRESHOLDS.keys())]

hex_sensitivity_pivot
